In [2]:
import numpy as np
from collections import Counter
from itertools import combinations, permutations
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)

ld = LammpsData.from_file('bulk_structure/mg2zn11_relaxed.data', atom_style='atomic')
alloy = AseAtomsAdaptor().get_atoms(ld.structure)
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")

Bulk: 39 atoms, Mg6Zn33


In [3]:
slab = surface(alloy, (1, 1, 0), 6, vacuum=8)
sc = make_supercell(slab, [[2,0,0],[0,2,0],[0,0,1]])

sym = np.array(sc.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')
z = sc.get_positions()[:, 2]
thick = z.max() - z.min()
area = np.linalg.norm(np.cross(sc.cell[0], sc.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(1,1,0), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(sc)}, Mg: {mg}, Zn: {zn}")
print(f"Thickness: {thick:.1f} A")
print(f"Area: {area:.1f} A²")
print(f"Stoichiometric: {'YES' if mg*11 == zn*2 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 936, Mg: 144, Zn: 792
Thickness: 37.0 A
Area: 429.2 A²
Stoichiometric: YES
Symmetric: False


In [4]:
z = sc.get_positions()[:, 2]
sym = np.array(sc.get_chemical_symbols())
order = np.argsort(z)

z_sorted = np.sort(z)
gaps = np.diff(z_sorted)
tol = max(0.02, min(0.3, np.median(gaps[gaps > 0.01]) / 2))
print(f"Tolerance: {tol:.3f} A")

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

n = len(layers)
z_min, z_max = z.min(), z.max()
print(f"Total layers: {n}")

print("\nSearching for symmetric removals...\n")

found_any = False
for bot_rm in range(0, min(15, n//2)):
    for top_rm in range(0, min(15, n//2)):
        if bot_rm == 0 and top_rm == 0:
            continue
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) < 100:
            continue
        tr = sc[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,1,0), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr.is_symmetric():
                removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                sym_tr = np.array(tr.get_chemical_symbols())
                mg_tr = sum(sym_tr == 'Mg')
                zn_tr = sum(sym_tr == 'Zn')
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                      f"Mg{mg_tr} Zn{zn_tr}, removed {removed_bot}+{removed_top}={removed_bot+removed_top}")
                found_any = True
        except:
            continue

if not found_any:
    print("No symmetric sub-slab found")

Tolerance: 0.112 A
Total layers: 121

Searching for symmetric removals...

  bot_rm=1, top_rm=1: 908 atoms, Mg144 Zn764, removed 20+8=28
  bot_rm=2, top_rm=2: 900 atoms, Mg144 Zn756, removed 24+12=36
  bot_rm=3, top_rm=3: 876 atoms, Mg144 Zn732, removed 36+24=60
  bot_rm=4, top_rm=4: 868 atoms, Mg136 Zn732, removed 40+28=68
  bot_rm=5, top_rm=5: 852 atoms, Mg136 Zn716, removed 48+36=84
  bot_rm=6, top_rm=6: 836 atoms, Mg136 Zn700, removed 56+44=100
  bot_rm=7, top_rm=7: 828 atoms, Mg128 Zn700, removed 60+48=108
  bot_rm=8, top_rm=8: 820 atoms, Mg128 Zn692, removed 64+52=116
  bot_rm=9, top_rm=9: 812 atoms, Mg128 Zn684, removed 68+56=124
  bot_rm=10, top_rm=10: 796 atoms, Mg128 Zn668, removed 76+64=140
  bot_rm=11, top_rm=11: 764 atoms, Mg112 Zn652, removed 92+80=172
  bot_rm=12, top_rm=12: 748 atoms, Mg112 Zn636, removed 100+88=188
  bot_rm=13, top_rm=13: 740 atoms, Mg112 Zn628, removed 104+92=196
  bot_rm=14, top_rm=14: 732 atoms, Mg112 Zn620, removed 108+96=204


In [6]:
removed_bot = layer_idx[0]
removed_top = layer_idx[n-1]
removed_all = removed_bot + removed_top

keep = []
for j in range(1, n - 1):
    keep.extend(layer_idx[j])

trimmed = sc[keep]
ps_trim = AseAtomsAdaptor().get_structure(trimmed)

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

rem_mg = sum(sym[j] == 'Mg' for j in removed_all)
rem_zn = sum(sym[j] == 'Zn' for j in removed_all)
print(f"\nRemoved: {len(removed_all)} atoms (Mg{rem_mg} Zn{rem_zn})")
print(f"Bottom: {len(removed_bot)}, Top: {len(removed_top)}")

# Find mutual pairs and unpaired
ps_orig = AseAtomsAdaptor().get_structure(sc)

paired_bot_top = []
for j in removed_bot:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    for k in removed_top:
        if np.linalg.norm(sc.positions[k] - inv_cart) < 0.5:
            paired_bot_top.append((j, k))
            break

used_bot = set(b for b, _ in paired_bot_top)
used_top = set(t for _, t in paired_bot_top)
unpaired_bot = [j for j in removed_bot if j not in used_bot]
unpaired_top = [j for j in removed_top if j not in used_top]

print(f"\nPaired bot<->top: {len(paired_bot_top)}")
print(f"Unpaired bot: {len(unpaired_bot)}, Unpaired top: {len(unpaired_top)}")

# Find mutual pairs among unpaired
all_unpaired = unpaired_bot + unpaired_top
used = set()
mutual = []
for i, j1 in enumerate(all_unpaired):
    if j1 in used:
        continue
    frac1 = ps_orig[j1].frac_coords
    inv1 = ps_orig.lattice.get_cartesian_coords((inv_translation - frac1) % 1.0)
    for j2 in all_unpaired[i+1:]:
        if j2 in used:
            continue
        if np.linalg.norm(sc.positions[j2] - inv1) < 0.5:
            mutual.append((j1, j2))
            used.add(j1)
            used.add(j2)
            break

remaining = [j for j in all_unpaired if j not in used]
print(f"Mutual pairs: {len(mutual)}")
print(f"Still unpaired: {len(remaining)}")

# Reconstruct: move one from each mutual pair
sc_fixed = sc.copy()
for keep_idx, move_idx in mutual:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    sc_fixed.positions[move_idx] = inv_cart

# If remaining unpaired, brute force
if len(remaining) > 0 and len(remaining) % 2 == 0:
    print(f"\nBrute force on {len(remaining)} remaining atoms...")
    
    rem_mg = [j for j in remaining if sym[j] == 'Mg']
    rem_zn = [j for j in remaining if sym[j] == 'Zn']
    
    found = False
    count = 0
    for keep_mg in combinations(rem_mg, len(rem_mg)//2) if rem_mg else [()]:
        move_mg = [j for j in rem_mg if j not in keep_mg]
        for keep_zn in combinations(rem_zn, len(rem_zn)//2) if rem_zn else [()]:
            move_zn = [j for j in rem_zn if j not in keep_zn]
            for perm_mg in permutations(keep_mg) if keep_mg else [()]:
                for perm_zn in permutations(keep_zn) if keep_zn else [()]:
                    sc_test = sc_fixed.copy()
                    for m, k in zip(move_mg, perm_mg):
                        frac = ps_orig[k].frac_coords
                        inv_frac = (inv_translation - frac) % 1.0
                        inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
                        sc_test.positions[m] = inv_cart
                    for m, k in zip(move_zn, perm_zn):
                        frac = ps_orig[k].frac_coords
                        inv_frac = (inv_translation - frac) % 1.0
                        inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
                        sc_test.positions[m] = inv_cart
                    
                    count += 1
                    if count % 5000 == 0:
                        print(f"  tested {count}...")
                    try:
                        ps_test = AseAtomsAdaptor().get_structure(sc_test)
                        slab_test = Slab(ps_test.lattice, ps_test.species, ps_test.frac_coords,
                            miller_index=(1,1,0), oriented_unit_cell=ps_test, shift=0,
                            scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
                        if slab_test.is_symmetric():
                            print(f"  FOUND after {count} tries!")
                            sc_fixed = sc_test
                            found = True
                            break
                    except:
                        continue
                if found:
                    break
            if found:
                break
        if found:
            break

sym_f = np.array(sc_fixed.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
    miller_index=(1,1,0), oriented_unit_cell=ps_fixed, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

print(f"\nAtoms: {len(sc_fixed)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if mg_f*11 == zn_f*2 else 'NO'}")
print(f"Symmetric: {slab_fixed.is_symmetric()}")
print(f"Thickness: {thick:.1f} A")
print(f"Area: {area:.1f} A²")

SG = P2/m
Inversion: f -> [0. 0. 0.] - f

Removed: 28 atoms (Mg0 Zn28)
Bottom: 20, Top: 8

Paired bot<->top: 8
Unpaired bot: 12, Unpaired top: 0
Mutual pairs: 0
Still unpaired: 12

Brute force on 12 remaining atoms...
  FOUND after 1 tries!

Atoms: 936, Mg: 144, Zn: 792
Stoichiometric: YES
Symmetric: True
Thickness: 37.0 A
Area: 429.2 A²


In [7]:
view(sc_fixed)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [9]:
ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg2zn11_110_6layers_reconstructed_ase.data")

print(f"Saved: slabs/mg2zn11_110_6layers_2x2_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg2zn11_110_6layers_2x2_reconstructed_ase.data
  Atoms: 936
  Thickness: 37.0 A
  Area: 429.2 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [10]:
ld_check = LammpsData.from_file("slabs/mg2zn11_110_6layers_reconstructed_ase.data", atom_style="atomic")
print(ld_check.structure.lattice)
print(f"\nComposition: {ld_check.structure.composition}")

# Also check what pymatgen assigned as type 1 and type 2
for i, site in enumerate(ld_check.structure[:3]):
    print(f"  Site {i}: {site.specie}")

24.636989 0.000000 0.000000
0.000000 17.420983 0.000000
0.000000 0.000000 52.955484

Composition: Zn792 Mg144
  Site 0: Zn
  Site 1: Zn
  Site 2: Mg


In [12]:
ld_bulk = LammpsData.from_file("bulk_structure/mg2zn11_relaxed.data", atom_style="atomic")
print("\nBulk lattice:")
print(ld_bulk.structure.lattice)
print(f"Composition: {ld_bulk.structure.composition}")


Bulk lattice:
8.710491 0.000000 0.000000
0.000000 8.710491 0.000000
0.000000 0.000000 8.710491
Composition: Zn33 Mg6


In [13]:
#unrelaxed surface energy calculation
E_slab = -1018.4585       # eV
E_bulk_per_fu =  -44.119792 / 3  # eV per formula unit 
n = 936 / 13                 # formula units in slab = 32
A = 429.2             # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.706597 eV
n * E_bulk = -1058.87501 eV
E_slab - n*E_bulk = 40.41651 eV
S = 0.047084 eV/Ang^2
S = 0.7544 J/m^2


In [15]:
#relaxed surface energy calculation
E_slab_relaxed = -1025.01581577042    # eV
E_bulk_per_fu = -44.119792 / 3  # eV per formula unit
n = 936 / 13                      # 32 formula units
A = 429.2                 # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 =  0.7544
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 33.85919 eV
S relaxed = 0.039445 eV/Ang^2
S relaxed = 0.6320 J/m^2

Unrelaxed S = 0.7544 J/m^2
Relaxed S   = 0.6320 J/m^2
Relaxation energy = 0.1224 J/m^2
